# English 5-Letter Word Ladder Training Data

Generate training examples for fine-tuning BERT on Word Ladder (5-letter English words).

- **Playable words** (start/target): `data/islands/english_5_strict_largest_island.txt`
- **Full vocab** (allowed steps): `data/islands/english_5_largest_island.txt`
- **Format**: binary classification — given (current, target), is candidate the correct next step?
- **Output**: train/val/test CSV in `data/training/`
- **Improvements**: split by (start, target) to avoid leakage; stratified by path length; deduplicated

In [2]:
import random
from pathlib import Path

import networkx as nx
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

SEED = 42
random.seed(SEED)

BASE = Path("../data")
ISLANDS = BASE / "islands"
TRAINING = BASE / "training"
TRAINING.mkdir(parents=True, exist_ok=True)

PLAYABLE_STRICT = ISLANDS / "english_5_strict_largest_island.txt"
FULL_VOCAB = ISLANDS / "english_5_largest_island.txt"
SEP = " [SEP] "


def load_words(path: Path) -> set[str]:
    """Load words from file, one per line."""
    return {w.strip().lower() for w in path.read_text(encoding="utf-8").splitlines() if w.strip()}


def one_letter_neighbors(w: str, vocab: set[str]) -> set[str]:
    """Words that differ from w by exactly one letter."""
    import string
    neighbors = set()
    for i in range(len(w)):
        for c in string.ascii_lowercase:
            if c != w[i]:
                cand = w[:i] + c + w[i + 1:]
                if cand in vocab:
                    neighbors.add(cand)
    return neighbors


# 1. Load playable strict words and full vocab
playable = load_words(PLAYABLE_STRICT)
vocab = load_words(FULL_VOCAB)
assert playable.issubset(vocab), "Playable must be subset of vocab"
print(f"Playable: {PLAYABLE_STRICT.name} ({len(playable)} words)")
print(f"Full vocab: {FULL_VOCAB.name} ({len(vocab)} words)")

Playable: english_5_strict_largest_island.txt (5330 words)
Full vocab: english_5_largest_island.txt (9902 words)


In [3]:
# 2. Build graph from full vocab
G = nx.Graph()
G.add_nodes_from(vocab)
for w in tqdm(vocab, desc="Building graph"):
    for nb in one_letter_neighbors(w, vocab):
        if w < nb:
            G.add_edge(w, nb)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Building graph: 100%|██████████| 9902/9902 [00:01<00:00, 9619.97it/s] 

Nodes: 9902, Edges: 34204


In [4]:
# 3. Sample random (start, target) pairs from playable words
TARGET_PAIRS = 18_000  # aim for 12k–20k valid paths, oversample to account for skips
playable_list = list(playable)
pairs = []
seen = set()
while len(pairs) < TARGET_PAIRS:
    s, t = random.sample(playable_list, 2)
    if s != t and (s, t) not in seen:
        seen.add((s, t))
        pairs.append((s, t))
random.shuffle(pairs)
print(f"Sampled {len(pairs)} unique (start, target) pairs")

Sampled 18000 unique (start, target) pairs


In [5]:
# 4. Find shortest paths, filter by steps (3–10 steps = 4–11 nodes)
MIN_STEPS = 3
MAX_STEPS = 10

valid_paths = []
for start, target in tqdm(pairs, desc="Finding paths"):
    try:
        path = nx.shortest_path(G, start, target)
    except nx.NetworkXNoPath:
        continue
    steps = len(path) - 1
    if MIN_STEPS <= steps <= MAX_STEPS:
        valid_paths.append(path)

print(f"Valid paths ({MIN_STEPS}–{MAX_STEPS} steps): {len(valid_paths)}")

Finding paths: 100%|██████████| 18000/18000 [00:10<00:00, 1761.14it/s]

Valid paths (3–10 steps): 15928


In [ ]:
# 5. Create examples from each path
# For each position i: current=path[i], target=path[-1], good_next=path[i+1]
# Positive: (current, target, good_next) -> label=1
# Negative: 3 per positive — wrong neighbors, or random from vocab
# Store start, target, path_len for split-by-pair and stratification

MULTI_POSITIVE = True  # True = all valid next steps per (current,target)
HARD_NEGATIVES = True  # True = wrong neighbors ranked by shared letters with target
NEGATIVES_PER_POSITIVE = 3
vocab_list = list(vocab)

def letters_in_common(a, b):
    return sum(1 for i in range(len(a)) if a[i] == b[i])

examples = []
for path in tqdm(valid_paths, desc="Creating examples"):
    start, target = path[0], path[-1]
    path_len = len(path)
    for i in range(len(path) - 1):
        current = path[i]
        text_a = current + SEP + target
        neighbors = one_letter_neighbors(current, vocab)

        if MULTI_POSITIVE:
            dist_curr = nx.shortest_path_length(G, current, target)
            valid_nexts = [n for n in neighbors if nx.shortest_path_length(G, n, target) == dist_curr - 1]
        else:
            valid_nexts = [path[i + 1]]

        for good_next in valid_nexts:
            examples.append({
                "text_a": text_a, "text_b": good_next, "label": 1,
                "start": start, "target": target, "path_len": path_len,
            })

        wrong_neighbors = [w for w in neighbors if w not in valid_nexts]
        if HARD_NEGATIVES:
            wrong_neighbors.sort(key=lambda w: -letters_in_common(w, target))
        neg_candidates = list(wrong_neighbors)
        if len(neg_candidates) < NEGATIVES_PER_POSITIVE:
            extra = [w for w in vocab_list if w not in valid_nexts and w != current and w not in neg_candidates]
            random.shuffle(extra)
            neg_candidates.extend(extra[: NEGATIVES_PER_POSITIVE - len(neg_candidates)])
        for neg in neg_candidates[:NEGATIVES_PER_POSITIVE]:
            examples.append({
                "text_a": text_a, "text_b": neg, "label": 0,
                "start": start, "target": target, "path_len": path_len,
            })

# 6. Deduplicate by (text_a, text_b, label) — keep first occurrence
seen = set()
deduped = []
for ex in examples:
    key = (ex["text_a"], ex["text_b"], ex["label"])
    if key not in seen:
        seen.add(key)
        deduped.append(ex)
examples = deduped
print(f"After dedup: {len(examples)} examples")

# Cap at 50k (raised for better performance; run on Colab)
MAX_EXAMPLES = 50_000
if len(examples) > MAX_EXAMPLES:
    random.shuffle(examples)
    examples = examples[:MAX_EXAMPLES]
    print(f"Capped to {len(examples)} examples")

Creating examples: 100%|██████████| 15928/15928 [01:07<00:00, 235.15it/s]


After dedup: 406806 examples
Capped to 15000 examples


In [7]:
# 7. Split by (start, target) — all examples from a pair go to same split (no leakage)
# Stratify by path length bin (manual split, no sklearn required)
pair_to_pathlen = {(ex["start"], ex["target"]): ex["path_len"] for ex in examples}
pairs_list = list(pair_to_pathlen.keys())
path_lens = [pair_to_pathlen[p] for p in pairs_list]

# Bin path length: short (4-5 nodes), medium (6-7), long (8-11)
path_bins = [0 if pl <= 5 else (1 if pl <= 7 else 2) for pl in path_lens]

# Group pairs by bin
from collections import defaultdict
bins_to_pairs = defaultdict(list)
for p, b in zip(pairs_list, path_bins):
    bins_to_pairs[b].append(p)

# Stratified split: 90% train, 5% val, 5% test per bin
train_pairs_set = set()
val_pairs_set = set()
test_pairs_set = set()
for bin_id, bin_pairs in bins_to_pairs.items():
    random.shuffle(bin_pairs)
    n_bin = len(bin_pairs)
    t1 = int(0.90 * n_bin)
    t2 = int(0.95 * n_bin)
    train_pairs_set.update(bin_pairs[:t1])
    val_pairs_set.update(bin_pairs[t1:t2])
    test_pairs_set.update(bin_pairs[t2:])

train_data = [ex for ex in examples if (ex["start"], ex["target"]) in train_pairs_set]
val_data = [ex for ex in examples if (ex["start"], ex["target"]) in val_pairs_set]
test_data = [ex for ex in examples if (ex["start"], ex["target"]) in test_pairs_set]

# 8. Save as CSV (columns: text_a, text_b, label)
train_df = pd.DataFrame([{"text_a": ex["text_a"], "text_b": ex["text_b"], "label": ex["label"]} for ex in train_data])
val_df = pd.DataFrame([{"text_a": ex["text_a"], "text_b": ex["text_b"], "label": ex["label"]} for ex in val_data])
test_df = pd.DataFrame([{"text_a": ex["text_a"], "text_b": ex["text_b"], "label": ex["label"]} for ex in test_data])

train_df.to_csv(TRAINING / "wordladder_english5_train.csv", index=False)
val_df.to_csv(TRAINING / "wordladder_english5_val.csv", index=False)
test_df.to_csv(TRAINING / "wordladder_english5_test.csv", index=False)

# 9. Print stats
n = len(examples)
pos_train = sum(ex["label"] for ex in train_data)
neg_train = len(train_data) - pos_train
pos_val = sum(ex["label"] for ex in val_data)
pos_test = sum(ex["label"] for ex in test_data)

print(f"\n=== Final stats ===")
print(f"Valid pairs: {len(valid_paths)}")
print(f"Unique (start,target) in data: {len(pairs_list)}")
print(f"Total examples: {n}")
print(f"Train: {len(train_data)} (pos: {pos_train}, neg: {neg_train})")
print(f"Val: {len(val_data)} (pos: {pos_val})")
print(f"Test: {len(test_data)} (pos: {pos_test})")
print(f"Class balance (train): {pos_train/len(train_data):.2%} pos, {neg_train/len(train_data):.2%} neg")
print(f"Saved to {TRAINING}/ (CSV)")


=== Final stats ===
Valid pairs: 15928
Unique (start,target) in data: 9479
Total examples: 15000
Train: 13444 (pos: 3385, neg: 10059)
Val: 785 (pos: 206)
Test: 771 (pos: 173)
Class balance (train): 25.18% pos, 74.82% neg
Saved to ..\data\training/ (CSV)
